In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

DATA_DIR = Path("..") / "data" / "processed"
val = pd.read_csv(DATA_DIR / "val.csv")
test_pub = pd.read_csv(DATA_DIR / "test_pub.csv")
LABEL_NAMES = {0: "Background", 1: "Basis", 2: "Discuss", 3: "Differ", 4: "Support"}

In [4]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)
from datasets import Dataset
import torch

def find_best_checkpoint(output_dir):
    ckpts = sorted(Path(output_dir).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint found in {output_dir}")
    return str(ckpts[-1])

def get_probs(base_model_name, output_dir, df):
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<CITE>", "[REF]"]})

    ckpt = find_best_checkpoint(output_dir)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt)

    def tok_fn(batch):
        return tokenizer(batch["citation_context"], truncation=True, max_length=128)

    ds = Dataset.from_pandas(df[["citation_context"]]).map(tok_fn, batched=True)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    args = TrainingArguments(output_dir="../models/_eval_tmp", per_device_eval_batch_size=32, report_to="none")
    trainer = Trainer(model=model, args=args, data_collator=collator)

    logits = trainer.predict(ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return probs

In [5]:
MODELS = {
    "berturk": ("dbmdz/bert-base-turkish-cased", "../models/berturk_citation_intent"),
    "modernbert": ("ytu-ce-cosmos/modernbert-tr-base-1k", "../models/modernbert_tr_citation_intent"),
    "electra": ("dbmdz/electra-base-turkish-mc4-cased-discriminator", "../models/electra_tr_citation_intent"),
}

val_probs = {}
test_probs = {}
for name, (base_model, out_dir) in MODELS.items():
    print(f"Loading {name}...")
    val_probs[name] = get_probs(base_model, out_dir, val)
    test_probs[name] = get_probs(base_model, out_dir, test_pub)

Loading berturk...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/550 [00:00<?, ? examples/s]

...

Loading electra...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/550 [00:00<?, ? examples/s]

In [6]:
def report(name, y_true, preds):
    print(f"=== {name} ===")
    print("Accuracy:", accuracy_score(y_true, preds))
    print("Macro F1:", f1_score(y_true, preds, average="macro"))
    print(classification_report(y_true, preds, target_names=list(LABEL_NAMES.values()), digits=3))

y_test = test_pub["citation_intent"]

for name, probs in test_probs.items():
    report(name, y_test, probs.argmax(axis=1))

avg_probs_test = np.mean(list(test_probs.values()), axis=0)
report("Soft-vote (avg probs)", y_test, avg_probs_test.argmax(axis=1))

from scipy.stats import mode
hard_votes = np.stack([p.argmax(axis=1) for p in test_probs.values()], axis=1)
hard_vote_preds = mode(hard_votes, axis=1, keepdims=False).mode
report("Hard-vote (majority)", y_test, hard_vote_preds)

=== berturk ===
Accuracy: 0.8727272727272727
Macro F1: 0.6582025955760133
              precision    recall  f1-score   support

  Background      0.898     0.960     0.928       420
       Basis      0.836     0.689     0.756        74
     Discuss      0.857     0.400     0.545        30
      Differ      0.500     0.500     0.500        10
     Support      0.562     0.562     0.562        16

    accuracy                          0.873       550
   macro avg      0.731     0.622     0.658       550
weighted avg      0.870     0.873     0.865       550

=== modernbert ===
Accuracy: 0.8581818181818182
Macro F1: 0.6713535273378631
              precision    recall  f1-score   support

  Background      0.915     0.921     0.918       420
       Basis      0.705     0.743     0.724        74
     Discuss      0.923     0.400     0.558        30
      Differ      0.545     0.600     0.571        10
     Support      0.480     0.750     0.585        16

    accuracy                      

In [7]:
from sklearn.linear_model import LogisticRegression

model_order = list(MODELS.keys())
X_val = np.concatenate([val_probs[m] for m in model_order], axis=1)   # (n_val, 15)
X_test = np.concatenate([test_probs[m] for m in model_order], axis=1)

meta = LogisticRegression(max_iter=1000, class_weight="balanced")
meta.fit(X_val, val["citation_intent"])
stacked_preds = meta.predict(X_test)

report("Stacked meta-learner (LogReg)", y_test, stacked_preds)

=== Stacked meta-learner (LogReg) ===
Accuracy: 0.8290909090909091
Macro F1: 0.615569701196985
              precision    recall  f1-score   support

  Background      0.923     0.881     0.901       420
       Basis      0.671     0.743     0.705        74
     Discuss      0.500     0.600     0.545        30
      Differ      0.250     0.500     0.333        10
     Support      0.727     0.500     0.593        16

    accuracy                          0.829       550
   macro avg      0.614     0.645     0.616       550
weighted avg      0.848     0.829     0.836       550



In [ ]:
from tabpfn import TabPFNClassifier
import torch
from dotenv import load_dotenv
load_dotenv()

meta_tabpfn = TabPFNClassifier(device="cuda", random_state=42)
meta_tabpfn.fit(X_val, val["citation_intent"].to_numpy())
tabpfn_preds = meta_tabpfn.predict(X_test)

report("Stacked meta-learner (TabPFN-3)", y_test, tabpfn_preds)

TabPFN device: cuda
=== Stacked meta-learner (TabPFN-3) ===
Accuracy: 0.8727272727272727
Macro F1: 0.5507154112327878
              precision    recall  f1-score   support

  Background      0.885     0.990     0.935       420
       Basis      0.936     0.595     0.727        74
     Discuss      0.923     0.400     0.558        30
      Differ      0.000     0.000     0.000        10
     Support      0.571     0.500     0.533        16

    accuracy                          0.873       550
   macro avg      0.663     0.497     0.551       550
weighted avg      0.869     0.873     0.858       550

